# Модуль 1. Разбор кейсов: Паспорт проекта и взаимодействие факторов

**Модуль 1. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** factorial.csv: четыре искусственных отклика при кодированных A/B; единицы условные.

Отделить исследовательский вопрос от инструмента и увидеть взаимодействие в плане 2×2.

**Workflow:** Вопрос → факторы → модель с взаимодействием → эффекты → ограничение насыщенного плана.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab01_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%201_SDLC.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [ ]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab01/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


## 1. Из таблицы эксперимента в матрицу модели

В лаборатории «Химера» четыре режима дали разные отклики. Сначала проверяем, какие сочетания факторов действительно представлены. Для кодированных A, B ∈ {−1,+1} модель y=β₀+βA·A+βB·B+βAB·AB имеет четыре столбца. Столбец AB нужен, когда действие A зависит от уровня B. Создание столбца ещё не доказывает наличие физического механизма.

**Перед запуском:** Прочитайте четыре строки и проверьте полный набор сочетаний ±1.

In [ ]:
design = pd.read_csv(DATA / "factorial.csv")
x = np.column_stack([np.ones(len(design)), design.A, design.B, design.A*design.B])


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Из таблицы эксперимента в матрицу модели — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Коэффициент и эффект — разные числа

Решение системы даёт коэффициенты в выбранной кодировке. При переходе A от −1 к +1 его вклад меняется на 2βA, поэтому эффект в два раза больше коэффициента. Контрольные числа относятся к искусственной таблице. Четыре параметра на четыре уникальные точки оставляют ноль остаточных степеней свободы: точное совпадение не является оценкой экспериментальной ошибки.

**Перед запуском:** Постройте матрицу [1,A,B,A·B], решите систему и сравните коэффициенты с эффектами.

In [ ]:
beta = np.linalg.solve(x, design.y)
effects = 2*beta[1:]
print("Коэффициенты:", beta, "Эффекты:", effects)
assert np.allclose(beta, [50,10,5,8])
assert np.allclose(effects, [20,10,16])
assert len(set(zip(design.A, design.B))) == 4


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Коэффициент и эффект — разные числа — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Взаимодействие на графике

Строим отдельную линию для каждого B. Непараллельные линии показывают, что один усреднённый эффект скрывает зависимость от второго фактора. График помогает поставить следующий опыт, но без повторов нельзя оценить шум и доверительный интервал. В паспорте решения фиксируем измеряемую величину, контроль и критерий остановки.

**Перед запуском:** Покажите две линии отклика и объясните, почему непараллельность означает взаимодействие.

In [ ]:
for b, group in design.groupby("B"):
    group = group.sort_values("A")
    plt.plot(group.A, group.y, "o-", label=f"B={b}")
plt.xlabel("A, кодированный уровень"); plt.ylabel("Отклик, условные единицы")
plt.legend(); plt.savefig(OUT / "factorial.png", dpi=150); plt.show()
artifact = {"beta": beta.tolist(), "effects": effects.tolist(), "residual_df": 0}


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Взаимодействие на графике — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

Численные эффекты 20/10/16; проверка покрытия четырёх комбинаций. Это тест арифметики, не экспериментальный эффект.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [ ]:
record={'module':1,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Карта чтения](../../Literature/READING_MAP.md).

Перейдите к [lab01_practice.ipynb](lab01_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.